# Milestone 2 & 3: LaTeX Content Parsing, BibTeX Extraction & Citation Matching Pipeline

Notebook này chạy đường ống xử lý toàn diện cho Milestone 2 & Milestone 3:
1. **Content Parser**: Đọc cấu trúc cây bài báo (`document` -> `section` -> `subsection` -> `sentence` / `equation` / `figure` / `table`).
2. **BibTeX Parser**: Bóc tách trích dẫn từ `.bib` và `\begin{thebibliography}` trong `.tex`, khử trùng lặp qua các phiên bản.
3. **Citation Matching**: Gán nhãn trích dẫn qua Semantic Scholar, sinh top-5 ứng viên dự đoán và chia tập train/valid/test.
4. **Xuất 6 bảng Parquet**: `papers`, `references`, `elements`, `hierarchy`, `bibtex`, `predictions`.

In [ ]:
#@title 1. Cài đặt thư viện & Mount Google Drive
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive connected!')
except Exception:
    print('Running locally or on Kaggle.')

# Install pyarrow & pandas if needed
!pip install -q pyarrow pandas

In [ ]:
#@title 2. Tải mã nguồn scraper & parser
!git clone -b feat/bibtex-parser-bao https://github.com/QuangPhat101/arXiv-Project.git /content/arXiv-Project
%cd /content/arXiv-Project
import sys
sys.path.append('/content/arXiv-Project')
print('Source code loaded successfully!')

In [ ]:
#@title 3. Chạy Pipeline trích xuất 6 bảng Parquet
from src.run_m2_pipeline import run_pipeline

# Cấu hình đường dẫn dữ liệu (tùy chỉnh theo thư mục của bạn)
DATA_DIR = Path('/content/drive/MyDrive/arxiv-m1-benchmark-20k/data') if Path('/content/drive/MyDrive/arxiv-m1-benchmark-20k/data').exists() else Path('/content/arXiv-Project/data')
OUTPUT_DIR = Path('/content/export_m2_m3')
WORKERS = 4  # Số luồng CPU Colab

print(f'Starting pipeline on: {DATA_DIR} -> output to: {OUTPUT_DIR}')
run_pipeline(data_dir=DATA_DIR, output_dir=OUTPUT_DIR, workers=WORKERS)

In [ ]:
#@title 4. Kiểm tra dữ liệu 6 bảng Parquet vừa xuất
import pyarrow.parquet as pq
import pandas as pd

out_dir = Path('/content/export_m2_m3')
for f in sorted(out_dir.glob('*.parquet')):
    t = pq.read_table(f)
    print(f'{f.name:<22}: {t.num_rows:>10,} rows | {t.num_columns:>2} cols | {f.stat().st_size / 1024 / 1024:>6.2f} MB')
    print('  Columns:', t.column_names)